# Natural Gas Winter Hedging Optimization (Gurobi)
**Team 28**: Shivani Murugapiran, Jake Deleray, James Wu, Yingshan Zhang, Jiayue Du

On October 1 a U.S. gas utility decides what share $h_m$ of each winter month's forecast demand
(Nov–Feb) to lock in at NYMEX futures prices; the rest is bought at the Henry Hub spot price.
This notebook builds every parameter from the files in `data/` (plus NOAA CPC heating degree days,
downloaded on first run) and solves the scenario LP with Gurobi for **Texas** and **Vermont**.

$$\min_{h,c}\ \sum_{s\in S}\pi_s c_s$$
$$c_s=\sum_{m\in M}\big[h_m D_m F_m+(D_{sm}-h_m D_m)\tilde P_{sm}\big]\quad\forall s\in S$$
$$L_m\le h_m\le U_m\quad\forall m\in M,\qquad \frac{\sum_{s\in H}\pi_s c_s}{\sum_{s\in H}\pi_s}\le C_{max}$$

where $\tilde P_{sm}=F_m\,P_{sm}/F_{sm}$. See the README for the full parameter table and assumptions.

**How to use:** edit the *Settings* cell below the setup cells, then *Run All*.
The notebook should sit in the repo root next to the `data/` folder.

## Setup
Imports, constants and the `Config` settings object.

In [ ]:
# !pip install -r requirements.txt   # uncomment on first run
from __future__ import annotations

import sys
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import gurobipy as gp
    from gurobipy import GRB
except ImportError as e:
    raise ImportError("gurobipy is required:  pip install gurobipy") from e

from IPython.display import Image, display

In [ ]:
ROOT = Path.cwd()  # run the notebook from the repo root
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"

MONTHS = ["Nov", "Dec", "Jan", "Feb"]
# Calendar month and year offset (relative to the winter's start year) for each delivery month
MONTH_CAL = {"Nov": (11, 0), "Dec": (12, 0), "Jan": (1, 1), "Feb": (2, 1)}
# On the first trading day of October, NYMEX contracts 1-4 are Nov, Dec, Jan, Feb
FUTURES_COL = {"Nov": "C1", "Dec": "C2", "Jan": "C3", "Feb": "C4"}

FIRST_WINTER = 1997  # 1997-98
LAST_WINTER = 2023   # 2023-24 (last October with futures quotes in the EIA file)

HEAT_CONTENT = 1.036  # MMBtu per Mcf (≈1,036 Btu/cf), so 1 MMcf = 1,036 MMBtu

STATE_NAMES = {"TX": "Texas", "VT": "Vermont", "NY": "New York"}
DEFAULT_COMPANY = {
    "TX": "ATMOS ENERGY CORPORATION",
    "VT": "VERMONT GAS SYSTEMS INC",
    "NY": "CONSOLIDATED EDISON NEW YORK INC",
}

EIA_CONS_URL = "https://www.eia.gov/dnav/ng/xls/NG_CONS_SUM_DCU_S{st}_M.xls"
NOAA_HDD_URL = (
    "https://ftp.cpc.ncep.noaa.gov/htdocs/degree_days/weighted/daily_data/"
    "{year}/StatesCONUS.Heating.txt"
)

In [ ]:
@dataclass
class Config:
    states: list[str] = field(default_factory=lambda: ["TX", "VT"])
    decision_year: int = 2023              # hedge decided Oct 1 of this year (2023 = backtest winter 2023-24)
    theta: float = 2.0                     # weight multiplier on H when g = 1
    gpr_percentile: float = 0.90           # GPR threshold percentile
    gpr_month: str | None = None           # month whose GPR sets g (default: Sept of decision_year)
    force_g: int | None = None             # override g (0/1)
    high_quantile: float = 0.75            # H = winters with avg spot >= this quantile
    high_basis: str = "rescaled"           # define H on rescaled spot (default) or "raw" spot
    cmax_factor: float | None = 1.0        # Cmax = factor * H-avg cost of the 50% hedge
    cmax_abs: float | None = None          # absolute Cmax in $ (overrides factor)
    lower: list[float] = field(default_factory=lambda: [0.0] * 4)
    upper: list[float] = field(default_factory=lambda: [1.0] * 4)
    futures: list[float] | None = None     # F_m for Nov, Dec, Jan, Feb on Oct 1 of decision_year
    share_years: list[int] | None = None   # EIA-176 years for the utility share (default: 5 before decision)
    companies: dict[str, str] = field(default_factory=dict)
    scenario_demand: str = "fitted"        # "fitted" (regression) or "actual"
    data_dir: Path = DATA_DIR
    out_dir: Path = OUT_DIR
    hdd_csv: Path | None = None
    download: bool = True
    sensitivity: bool = True
    plots: bool = True
    verbose: bool = False

## Settings
Edit these and *Run All*. Each one corresponds to a command-line flag of `gurobi.py`.

**Forecast vs. backtest.** `decision_year` is the year the hedge is decided (on October 1).
* `2023` (default): **backtest** of winter 2023-24, the latest winter fully covered by the data in this repo.
  The model sees only winters through 2022-23 and the Oct 2 2023 futures, then is scored against actual
  Nov 2023 – Feb 2024 spot prices and demand.
* `2026`: forecast for winter 2026-27 (no actual outcome yet).
* `2025`: **backtest** of winter 2025-26. The model only sees winters before Oct 2025, then the hedge is
  scored against the actual Nov 2025 – Feb 2026 spot prices and demand.
* `1998`–`2023`: backtest of any earlier winter; the futures come from the EIA data automatically.

> **Futures strip:** EIA's NYMEX series ends Apr 5 2024, so for 2025 or 2026 enter the **Oct 1 CME settlements**
> for Nov / Dec / Jan / Feb in `futures`. A 2025 backtest stops with an error without them; a 2026 forecast
> falls back to the Oct 2 2023 strip with a warning.

In [ ]:
cfg = Config(
    states=["TX", "VT"],          # any of "TX", "VT", "NY"
    decision_year=2023,           # 2023 = backtest winter 2023-24 (all data available); 2025 / 2026 need futures
    futures=None,                 # Oct 1 settlements of decision_year, e.g. [3.10, 3.55, 3.90, 3.70]
    theta=2.0,                    # weight multiplier on high-price winters when g = 1
    cmax_factor=1.0,              # Cmax = factor x high-price avg cost of a 50% hedge; None = no limit
    cmax_abs=None,                # absolute Cmax in $ (overrides cmax_factor)
    lower=[0.0, 0.0, 0.0, 0.0],   # L_m
    upper=[1.0, 1.0, 1.0, 1.0],   # U_m
    gpr_month=None,               # None = September of decision_year
    force_g=None,                 # None = use GPR rule; 0 or 1 to override
    high_basis="rescaled",        # "rescaled" (default) or "raw" (Milestone 3 definition of H)
    scenario_demand="fitted",     # "fitted" regression or "actual" consumption
    companies={},                 # e.g. {"TX": "CENTERPOINT ENERGY ENTEX"}
    hdd_csv=None,                 # offline HDD file (columns state,date,hdd)
    download=True,                # fetch NOAA HDD / EIA state consumption if missing
    sensitivity=True,
    plots=True,
    verbose=False,                # True shows the Gurobi log
)
assert all(0 <= l <= u <= 1 for l, u in zip(cfg.lower, cfg.upper)), "need 0 <= L_m <= U_m <= 1"

### Helpers

In [ ]:
def _download(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  downloading {url}")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (hedging-model)"})
    with urllib.request.urlopen(req, timeout=60) as r:
        dest.write_bytes(r.read())
    return dest

In [ ]:
def _read_eia_sheet(path: Path, sheet: str = "Data 1") -> pd.DataFrame:
    """Read an EIA dnav workbook sheet: row 2 = source keys, row 3 = labels, data after."""
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    keys = raw.iloc[1].tolist()
    df = raw.iloc[3:].copy()
    df.columns = ["Date"] + [str(k) for k in keys[1:]]
    df["Date"] = pd.to_datetime(df["Date"])
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.reset_index(drop=True)

In [ ]:
def winter_label(y: int) -> str:
    return f"{y}-{str(y + 1)[-2:]}"

In [ ]:
def winter_month_date(y: int, m: str) -> tuple[int, int]:
    cal, off = MONTH_CAL[m]
    return y + off, cal

In [ ]:
def scenario_winters(cfg: Config) -> list[str]:
    """Winters usable as scenarios: only those that ended before the decision date."""
    last = min(LAST_WINTER, cfg.decision_year - 1)
    return [winter_label(y) for y in range(FIRST_WINTER, last + 1)]

In [ ]:
def gpr_month(cfg: Config) -> str:
    return cfg.gpr_month or f"{cfg.decision_year}-09"

## 1. Prices
Henry Hub spot $P_{sm}$ (monthly mean of daily prices) and NYMEX futures $F_{sm}$ on the first trading day of October (contracts 1–4 = Nov, Dec, Jan, Feb).

In [ ]:
def monthly_spot(cfg: Config) -> pd.Series:
    """Monthly average Henry Hub spot price indexed by (year, month)."""
    df = _read_eia_sheet(cfg.data_dir / "RNGWHHDd.xls")
    df = df.rename(columns={"RNGWHHD": "spot"}).dropna(subset=["spot"])
    return df.groupby([df.Date.dt.year, df.Date.dt.month])["spot"].mean()

In [ ]:
def actual_winter_spot(cfg: Config, y: int) -> pd.Series | None:
    """Realized monthly spot for winter y (Nov y .. Feb y+1), or None if not complete yet."""
    monthly = monthly_spot(cfg)
    vals = [monthly.get(winter_month_date(y, m)) for m in MONTHS]
    if any(v is None or pd.isna(v) for v in vals):
        return None
    return pd.Series(vals, index=MONTHS, dtype=float)

In [ ]:
def load_spot(cfg: Config) -> pd.DataFrame:
    """P_sm: monthly average of daily Henry Hub spot prices, indexed by (winter, month)."""
    monthly = monthly_spot(cfg)
    rows = {}
    for y in range(FIRST_WINTER, LAST_WINTER + 1):
        rows[winter_label(y)] = [monthly.loc[winter_month_date(y, m)] for m in MONTHS]
    return pd.DataFrame.from_dict(rows, orient="index", columns=MONTHS)

In [ ]:
def load_futures(cfg: Config) -> tuple[pd.DataFrame, pd.Series]:
    """F_sm: Nov-Feb futures on the first trading day of October of each winter.

    Returns (historical strips, the most recent October strip in the file)."""
    df = _read_eia_sheet(cfg.data_dir / "NG_PRI_FUT_S1_D.xls", sheet="Data 2")
    df = df.rename(columns={"RNGC1": "C1", "RNGC2": "C2", "RNGC3": "C3", "RNGC4": "C4"})
    rows, dates = {}, {}
    for y in range(FIRST_WINTER, LAST_WINTER + 1):
        sub = df[(df.Date >= f"{y}-10-01")].dropna(subset=["C1", "C2", "C3", "C4"]).head(1)
        if sub.empty:
            raise ValueError(f"No October {y} futures quotes found")
        rows[winter_label(y)] = [float(sub[FUTURES_COL[m]].iloc[0]) for m in MONTHS]
        dates[winter_label(y)] = sub.Date.iloc[0].date()
    fut = pd.DataFrame.from_dict(rows, orient="index", columns=MONTHS)
    fut.attrs["quote_dates"] = dates
    return fut, fut.iloc[-1]

## 2. Weather: NOAA CPC heating degree days
Daily population-weighted HDD by state (`StatesCONUS.Heating.txt`), cached in `data/hdd/`, summed to monthly $HDD_{sm}$.

In [ ]:
def _parse_cpc_file(text: str) -> pd.DataFrame:
    lines = [ln.strip().strip('"') for ln in text.splitlines() if ln.strip()]
    start = next(i for i, ln in enumerate(lines) if ln.startswith("Region|"))
    header = lines[start].split("|")
    recs = []
    for ln in lines[start + 1:]:
        parts = ln.split("|")
        if len(parts) != len(header):
            continue
        st = parts[0].strip()
        for d, v in zip(header[1:], parts[1:]):
            recs.append((st, d, v))
    out = pd.DataFrame(recs, columns=["state", "date", "hdd"])
    out["date"] = pd.to_datetime(out["date"], format="%Y%m%d")
    out["hdd"] = pd.to_numeric(out["hdd"], errors="coerce")
    out.loc[out.hdd < 0, "hdd"] = np.nan  # missing-value flags
    return out

In [ ]:
def load_hdd(cfg: Config, states: list[str]) -> pd.DataFrame:
    """Daily HDD (state, date, hdd) from NOAA CPC StatesCONUS.Heating.txt, cached in data/hdd/."""
    if cfg.hdd_csv is not None:
        df = pd.read_csv(cfg.hdd_csv, parse_dates=["date"])
        return df[df.state.isin(states)]
    cache = cfg.data_dir / "hdd"
    frames = []
    years = sorted(set(range(FIRST_WINTER, LAST_WINTER + 2))
                   | {cfg.decision_year, cfg.decision_year + 1})
    for year in years:
        f = cache / f"StatesCONUS.Heating.{year}.txt"
        if not f.exists():
            if not cfg.download:
                if year > LAST_WINTER + 1:
                    continue  # only needed for the backtest
                raise FileNotFoundError(f"{f} missing and --no-download set")
            try:
                _download(NOAA_HDD_URL.format(year=year), f)
            except Exception as e:
                if year > LAST_WINTER + 1:  # decision-winter weather may not exist yet
                    print(f"  (no NOAA HDD file for {year}: {e})")
                    continue
                raise
        frames.append(_parse_cpc_file(f.read_text()))
    df = pd.concat(frames, ignore_index=True)
    return df[df.state.isin(states)]

In [ ]:
def monthly_hdd(hdd: pd.DataFrame, st: str) -> pd.DataFrame:
    """HDD_sm: sum of daily HDD within each delivery month of each winter."""
    d = hdd[hdd.state == st]
    tot = d.groupby([d.date.dt.year, d.date.dt.month])["hdd"].sum(min_count=1)
    rows = {}
    for y in range(FIRST_WINTER, LAST_WINTER + 1):
        rows[winter_label(y)] = [tot.get(winter_month_date(y, m), np.nan) for m in MONTHS]
    out = pd.DataFrame.from_dict(rows, orient="index", columns=MONTHS)
    if out.isna().any().any():
        missing = out[out.isna().any(axis=1)].index.tolist()
        raise ValueError(f"Missing HDD for {st} in winters {missing}")
    return out

## 3. Demand
State residential + commercial consumption (MMcf → MMBtu at 1,036 MMBtu/MMcf), the utility's share of state deliveries from EIA-176, and the monthly regression $D_{state,sm}=a_m+b_m\,HDD_{sm}$.

In [ ]:
def state_rc_monthly(cfg: Config, st: str) -> pd.Series:
    """State residential + commercial consumption in MMBtu, indexed by (year, month)."""
    f = cfg.data_dir / f"NG_CONS_SUM_DCU_S{st}_M.xls"
    if not f.exists():
        if not cfg.download:
            raise FileNotFoundError(f"{f} missing and --no-download set")
        _download(EIA_CONS_URL.format(st=st), f)
    df = _read_eia_sheet(f)
    res = next(c for c in df.columns if c.startswith("N3010"))
    com = next(c for c in df.columns if c.startswith("N3020"))
    df["rc_mmbtu"] = (df[res] + df[com]) * 1_000 * HEAT_CONTENT  # MMcf -> Mcf -> MMBtu
    return df.set_index([df.Date.dt.year, df.Date.dt.month])["rc_mmbtu"]

In [ ]:
def load_state_consumption(cfg: Config, st: str) -> pd.DataFrame:
    """Residential + commercial consumption for (winter, month), in MMBtu."""
    tot = state_rc_monthly(cfg, st)
    rows = {}
    for y in range(FIRST_WINTER, LAST_WINTER + 1):
        rows[winter_label(y)] = [tot.loc[winter_month_date(y, m)] for m in MONTHS]
    return pd.DataFrame.from_dict(rows, orient="index", columns=MONTHS)

In [ ]:
def utility_share(cfg: Config, st: str) -> tuple[str, float]:
    """Utility's share of state residential + commercial deliveries (EIA-176)."""
    df = pd.read_excel(cfg.data_dir / "eia176_company_deliveries.xlsx")
    df.columns = [int(c) if isinstance(c, (int, float)) else c for c in df.columns]
    df["Company"] = df["Company"].astype(str).str.strip()
    company = cfg.companies.get(st, DEFAULT_COMPANY.get(st))
    sub = df[(df.Area == STATE_NAMES[st]) & df.Item.isin(["Residential Volume", "Commercial Volume"])]
    wanted = cfg.share_years or list(range(cfg.decision_year - 5, cfg.decision_year))
    years = [y for y in wanted if y in sub.columns]
    if not years:  # decision year beyond the EIA-176 data: use the latest 5 years available
        years = sorted(c for c in sub.columns if isinstance(c, int))[-5:]
    by_co = sub.groupby("Company")[years].sum(min_count=1)
    total = by_co.loc["Total of All Companies"]
    if company not in by_co.index:
        # fall back to the largest LDC in the state
        company = by_co.drop("Total of All Companies").mean(axis=1).idxmax()
    share = float((by_co.loc[company] / total).mean())
    return company, share

In [ ]:
@dataclass
class DemandFit:
    a: pd.Series
    b: pd.Series
    r2: pd.Series
    hdd_normal: pd.Series

In [ ]:
def fit_demand(state_mmbtu: pd.DataFrame, hdd: pd.DataFrame) -> DemandFit:
    """OLS per month: StateDemand_sm = a_m + b_m * HDD_sm."""
    a, b, r2 = {}, {}, {}
    for m in MONTHS:
        x, y = hdd[m].values.astype(float), state_mmbtu[m].values.astype(float)
        bm, am = np.polyfit(x, y, 1)
        resid = y - (am + bm * x)
        a[m], b[m] = am, bm
        r2[m] = 1 - resid.var() / y.var()
    return DemandFit(pd.Series(a), pd.Series(b), pd.Series(r2), hdd.mean())

## 4. Geopolitical risk indicator
$g=1$ if the GPR value for `gpr_month` is at or above the 90th percentile of monthly GPR since 1985.

In [ ]:
def geopolitical_indicator(cfg: Config) -> tuple[int, float, float]:
    g = pd.read_excel(cfg.data_dir / "data_gpr_export.xls", usecols=["month", "GPR"])
    g["month"] = pd.to_datetime(g["month"])
    g = g.dropna(subset=["GPR"])
    hist = g[g.month >= "1985-01-01"]["GPR"]
    thresh = float(hist.quantile(cfg.gpr_percentile))
    cur = g[g.month.dt.strftime("%Y-%m") == gpr_month(cfg)]["GPR"]
    if cur.empty:
        raise ValueError(f"No GPR value for {gpr_month(cfg)}")
    value = float(cur.iloc[0])
    flag = int(value >= thresh) if cfg.force_g is None else int(cfg.force_g)
    return flag, value, thresh

## 5. Assemble the model inputs for one state
Builds $F_m, \tilde P_{sm}, D_m, D_{sm}, H$ and $\pi_s$.

In [ ]:
@dataclass
class Instance:
    state: str
    company: str
    share: float
    F: pd.Series            # F_m
    F_hist: pd.DataFrame    # F_sm
    P: pd.DataFrame         # P_sm
    Ptilde: pd.DataFrame    # rescaled spot
    D: pd.Series            # D_m
    Ds: pd.DataFrame        # D_sm
    HDD: pd.DataFrame
    fit: DemandFit
    H: list[str]
    pi: pd.Series
    g: int
    gpr_value: float
    gpr_thresh: float

In [ ]:
def build_instance(cfg: Config, st: str, spot, fut_hist, F_now, hdd_daily, gpr) -> Instance:
    company, share = utility_share(cfg, st)
    S = scenario_winters(cfg)  # no look-ahead: only winters that ended before the decision
    spot, fut_hist = spot.loc[S], fut_hist.loc[S]
    state_d = load_state_consumption(cfg, st).loc[S]
    HDD = monthly_hdd(hdd_daily, st).loc[S]
    fit = fit_demand(state_d, HDD)

    D = share * (fit.a + fit.b * fit.hdd_normal)
    if cfg.scenario_demand == "fitted":
        Ds = share * (HDD * fit.b + fit.a)
    else:
        Ds = share * state_d
    Ds = Ds.clip(lower=0)

    Ptilde = (spot / fut_hist) * F_now

    basis = spot if cfg.high_basis == "raw" else Ptilde
    avg = basis.mean(axis=1)
    H = avg[avg >= avg.quantile(cfg.high_quantile)].index.tolist()

    g, gv, gt = gpr
    pi = pd.Series(1.0, index=spot.index)
    if g == 1:
        pi[H] *= cfg.theta
    pi /= pi.sum()

    return Instance(st, company, share, F_now, fut_hist, spot, Ptilde, D, Ds, HDD, fit, H, pi, g, gv, gt)

## 6. Cost evaluation and the Gurobi model
`solve()` is the LP: variables $h_m, c_s$, the scenario cost constraints, hedge bounds, and the high-price cost limit.

In [ ]:
def scenario_costs(inst: Instance, h: pd.Series) -> pd.Series:
    """c_s for a fixed hedge vector h (used for benchmarks and Cmax calibration)."""
    hedged = (h * inst.D * inst.F).sum()
    spot_part = ((inst.Ds - h * inst.D) * inst.Ptilde).sum(axis=1)
    return hedged + spot_part

In [ ]:
def high_avg(inst: Instance, c: pd.Series) -> float:
    w = inst.pi[inst.H]
    return float((w * c[inst.H]).sum() / w.sum())

In [ ]:
def resolve_cmax(cfg: Config, inst: Instance) -> float | None:
    if cfg.cmax_abs is not None:
        return cfg.cmax_abs
    return cmax_from_factor(inst, cfg.cmax_factor)

In [ ]:
def cmax_from_factor(inst: Instance, factor: float | None) -> float | None:
    """Cmax = factor x (high-price average cost of hedging 50% every month)."""
    if factor is None:
        return None
    ref = high_avg(inst, scenario_costs(inst, pd.Series(0.5, index=MONTHS)))
    return factor * ref

In [ ]:
@dataclass
class Solution:
    status: str
    h: pd.Series | None = None
    c: pd.Series | None = None
    expected: float | None = None
    high_avg: float | None = None
    cmax: float | None = None
    cmax_dual: float | None = None

In [ ]:
def solve(cfg: Config, inst: Instance, cmax: float | None) -> Solution:
    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 1 if cfg.verbose else 0)
    env.start()
    mdl = gp.Model(f"hedge_{inst.state}", env=env)

    S = list(inst.pi.index)
    h = mdl.addVars(MONTHS, lb={m: cfg.lower[i] for i, m in enumerate(MONTHS)},
                    ub={m: cfg.upper[i] for i, m in enumerate(MONTHS)}, name="h")
    c = mdl.addVars(S, lb=-GRB.INFINITY, name="c")

    # (1) scenario cost definition
    for s in S:
        mdl.addConstr(
            c[s] == gp.quicksum(
                h[m] * inst.D[m] * inst.F[m]
                + (inst.Ds.at[s, m] - h[m] * inst.D[m]) * inst.Ptilde.at[s, m]
                for m in MONTHS),
            name=f"cost[{s}]")

    # (2) hedge-share bounds are variable bounds (L_m <= h_m <= U_m)

    # (3) high-price scenario cost limit
    risk = None
    if cmax is not None:
        wH = float(inst.pi[inst.H].sum())
        risk = mdl.addConstr(
            gp.quicksum(inst.pi[s] * c[s] for s in inst.H) <= cmax * wH, name="high_price_limit")

    mdl.setObjective(gp.quicksum(inst.pi[s] * c[s] for s in S), GRB.MINIMIZE)
    mdl.optimize()

    if mdl.Status != GRB.OPTIMAL:
        status = {GRB.INFEASIBLE: "infeasible", GRB.INF_OR_UNBD: "infeasible"}.get(
            mdl.Status, f"status_{mdl.Status}")
        mdl.dispose(); env.dispose()
        return Solution(status, cmax=cmax)

    hv = pd.Series({m: h[m].X for m in MONTHS})
    cv = pd.Series({s: c[s].X for s in S})
    dual = None
    if risk is not None:
        # $ of expected cost saved per extra $ of allowed high-price average cost
        dual = -risk.Pi * float(inst.pi[inst.H].sum())
    sol = Solution("optimal", hv, cv, mdl.ObjVal, high_avg(inst, cv), cmax, dual)
    mdl.dispose(); env.dispose()
    return sol

## 7. Reporting, benchmarks and sensitivity
Strategy comparison (0–100% and optimal hedges), the θ × $C_{max}$ sensitivity grid, plots and CSV outputs.

In [ ]:
def strategy_table(inst: Instance, sol: Solution) -> pd.DataFrame:
    strategies = {f"{int(k * 100)}% hedge": pd.Series(k, index=MONTHS) for k in (0, .25, .5, .75, 1)}
    if sol.h is not None:
        strategies["Optimal"] = sol.h
    total_vol = inst.D.sum()
    rows = []
    for name, h in strategies.items():
        c = scenario_costs(inst, h)
        rows.append({
            "strategy": name,
            **{f"h_{m}": round(h[m], 4) for m in MONTHS},
            "expected_cost": (inst.pi * c).sum(),
            "high_price_avg_cost": high_avg(inst, c),
            "worst_winter_cost": c.max(),
            "worst_winter": c.idxmax(),
            "std_cost": np.sqrt((inst.pi * (c - (inst.pi * c).sum()) ** 2).sum()),
            "expected_$per_MMBtu": (inst.pi * c).sum() / total_vol,
        })
    return pd.DataFrame(rows).set_index("strategy")

In [ ]:
def sensitivity(cfg: Config, inst: Instance) -> dict[str, pd.DataFrame]:
    """Re-solve over a grid of theta (H weight, as if g = 1; theta = 1 is the g = 0 case)
    and Cmax factors."""
    thetas = [1.0, 1.5, 2.0, 3.0, 4.0]
    factors = [0.90, 0.94, 0.97, 1.00, 1.03, 1.06, 1.10, None]
    exp_tab, hedge_tab = {}, {}
    for th in thetas:
        pi = pd.Series(1.0, index=inst.pi.index)
        pi[inst.H] *= th
        pi /= pi.sum()
        sub = Instance(**{**inst.__dict__, "pi": pi})
        exp_row, h_row = {}, {}
        for fct in factors:
            col = "no limit" if fct is None else f"{fct:.2f}x"
            cmax = cmax_from_factor(sub, fct)
            s = solve(cfg, sub, cmax)
            exp_row[col] = s.expected if s.status == "optimal" else np.nan
            h_row[col] = "/".join(f"{v:.2f}" for v in s.h) if s.h is not None else s.status
        exp_tab[f"theta={th}"] = exp_row
        hedge_tab[f"theta={th}"] = h_row
    return {
        "sensitivity_expected_cost": pd.DataFrame.from_dict(exp_tab, orient="index"),
        "sensitivity_hedge_shares": pd.DataFrame.from_dict(hedge_tab, orient="index"),
    }

In [ ]:
def make_plots(out: Path, inst: Instance, strat: pd.DataFrame, sens: dict | None):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, r in strat.iterrows():
        ax.scatter(r.expected_cost / 1e6, r.high_price_avg_cost / 1e6, s=60,
                   marker="*" if name == "Optimal" else "o", zorder=3)
        ax.annotate(name, (r.expected_cost / 1e6, r.high_price_avg_cost / 1e6),
                    textcoords="offset points", xytext=(6, 4), fontsize=9)
    ax.set_xlabel("Expected winter cost ($M)")
    ax.set_ylabel("Avg cost in high-price winters H ($M)")
    ax.set_title(f"{inst.state}: cost vs. high-price exposure")
    ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(out / "strategy_frontier.png", dpi=150); plt.close(fig)

    if sens:
        t = sens["sensitivity_expected_cost"] / 1e6
        fig, ax = plt.subplots(figsize=(9, 4))
        im = ax.imshow(t.values, cmap="viridis_r", aspect="auto")
        ax.set_xticks(range(t.shape[1]), t.columns, rotation=30)
        ax.set_yticks(range(t.shape[0]), t.index)
        for i in range(t.shape[0]):
            for j in range(t.shape[1]):
                v = t.values[i, j]
                ax.text(j, i, "infeas." if np.isnan(v) else f"{v:,.1f}", ha="center", va="center",
                        fontsize=8, color="white")
        ax.set_xlabel("Cmax (multiple of 50%-hedge high-price cost)")
        ax.set_title(f"{inst.state}: expected cost ($M) by theta and Cmax")
        fig.colorbar(im, ax=ax, shrink=.8)
        fig.tight_layout(); fig.savefig(out / "sensitivity_heatmap.png", dpi=150); plt.close(fig)

In [ ]:
def out_path(cfg: Config, st: str) -> Path:
    return cfg.out_dir / f"{st}_{cfg.decision_year}"

In [ ]:
def report(cfg: Config, inst: Instance, sol: Solution) -> None:
    out = out_path(cfg, inst.state)
    out.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 72}\n{STATE_NAMES[inst.state]} ({inst.state}): hedge decided Oct 1 "
          f"{cfg.decision_year} for winter {winter_label(cfg.decision_year)}\n{'=' * 72}")
    S = inst.pi.index
    print(f"Scenarios: {len(S)} winters, {S[0]} to {S[-1]}")
    print(f"Utility: {inst.company}  (share of state R+C deliveries = {inst.share:.1%})")
    print("Demand regression  D_state = a_m + b_m * HDD_m")
    print(pd.DataFrame({"a_m (MMBtu)": inst.fit.a, "b_m (MMBtu/HDD)": inst.fit.b,
                        "R^2": inst.fit.r2, "normal HDD": inst.fit.hdd_normal}).round(3).to_string())
    print(f"\nForecast utility demand D_m (MMBtu):\n{inst.D.round(0).to_string()}")
    print(f"\nHigh-price winters H ({len(inst.H)}): {', '.join(inst.H)}")
    print(f"GPR {gpr_month(cfg)} = {inst.gpr_value:.1f}, threshold = {inst.gpr_thresh:.1f} "
          f"-> g = {inst.g}" + (f", H weights x{cfg.theta}" if inst.g else ""))
    if sol.status != "optimal":
        print(f"\nModel status: {sol.status.upper()} (Cmax = {sol.cmax:,.0f}). "
              "Loosen --cmax-factor or the hedge bounds.")
        return
    print(f"\nOptimal hedge shares: " + ", ".join(f"{m} {sol.h[m]:.1%}" for m in MONTHS))
    print(f"Expected cost      : ${sol.expected:,.0f}")
    print(f"High-price avg cost: ${sol.high_avg:,.0f}"
          + (f"  (Cmax = ${sol.cmax:,.0f})" if sol.cmax else ""))
    if sol.cmax_dual is not None:
        print(f"Shadow price of Cmax: {sol.cmax_dual:.4f} $ expected / $ of Cmax")

    strat = strategy_table(inst, sol)
    print("\nStrategy comparison:")
    print(strat.drop(columns=[f"h_{m}" for m in MONTHS]).to_string(
        float_format=lambda v: f"{v:,.2f}" if abs(v) < 100 else f"{v:,.0f}"))

    # save parameters and results
    pd.DataFrame({"F_m": inst.F, "D_m": inst.D, "L_m": cfg.lower, "U_m": cfg.upper}).to_csv(
        out / "parameters_month.csv")
    pd.concat({"F_sm": inst.F_hist, "P_sm": inst.P, "Ptilde_sm": inst.Ptilde,
               "D_sm": inst.Ds, "HDD_sm": inst.HDD}, axis=1).assign(
        pi_s=inst.pi, in_H=inst.pi.index.isin(inst.H), c_s_optimal=sol.c).to_csv(
        out / "parameters_scenario.csv")
    pd.DataFrame({"h_m": sol.h, "hedged_MMBtu": sol.h * inst.D}).to_csv(out / "optimal_hedge.csv")
    strat.to_csv(out / "strategy_comparison.csv")

    sens = None
    if cfg.sensitivity:
        sens = sensitivity(cfg, inst)
        for k, v in sens.items():
            v.to_csv(out / f"{k}.csv")
        print("\nSensitivity - hedge shares Nov/Dec/Jan/Feb (rows theta, cols Cmax factor):")
        print(sens["sensitivity_hedge_shares"].to_string())
    if cfg.plots:
        make_plots(out, inst, strat, sens)
    print(f"\nResults written to {out}")

## 8. Backtest
When `decision_year` is a past year, score the hedge against the winter that actually happened: realized Henry Hub spot, actual demand (EIA consumption × utility share), versus fixed 0–100% hedges and a perfect-hindsight hedge.

In [ ]:
@dataclass
class Backtest:
    months: pd.DataFrame       # month-by-month prices, demand and hedge gains
    strategies: pd.DataFrame   # realized winter cost of each strategy
    demand_source: str

In [ ]:
def actual_winter_demand(cfg: Config, inst: Instance, hdd_daily: pd.DataFrame) -> tuple[pd.Series, str]:
    """Realized utility demand for the decision winter: actual EIA consumption x share if
    published, else the regression at actual HDD, else the normal-weather forecast."""
    y = cfg.decision_year
    try:
        tot = state_rc_monthly(cfg, inst.state)
        vals = [tot.get(winter_month_date(y, m)) for m in MONTHS]
        if all(v is not None and not pd.isna(v) for v in vals):
            return inst.share * pd.Series(vals, index=MONTHS, dtype=float), "actual EIA consumption"
    except Exception:
        pass
    d = hdd_daily[hdd_daily.state == inst.state]
    tot = d.groupby([d.date.dt.year, d.date.dt.month])["hdd"].sum(min_count=1)
    hdd = [tot.get(winter_month_date(y, m)) for m in MONTHS]
    if all(v is not None and not pd.isna(v) for v in hdd):
        hdd = pd.Series(hdd, index=MONTHS, dtype=float)
        return inst.share * (inst.fit.a + inst.fit.b * hdd), "regression at actual HDD"
    return inst.D.copy(), "normal-weather forecast (actual weather not available)"

In [ ]:
def backtest(cfg: Config, inst: Instance, sol: Solution, hdd_daily: pd.DataFrame) -> Backtest | None:
    P_act = actual_winter_spot(cfg, cfg.decision_year)
    if P_act is None or sol.h is None:
        return None
    D_act, source = actual_winter_demand(cfg, inst, hdd_daily)

    def realized(h: pd.Series) -> float:
        return float((h * inst.D * inst.F + (D_act - h * inst.D) * P_act).sum())

    lower = pd.Series(cfg.lower, index=MONTHS)
    upper = pd.Series(cfg.upper, index=MONTHS)
    strategies = {f"{int(k * 100)}% hedge": pd.Series(k, index=MONTHS) for k in (0, .25, .5, .75, 1)}
    strategies["Model (optimal)"] = sol.h
    strategies["Perfect hindsight"] = upper.where(inst.F < P_act, lower)
    base = realized(strategies["0% hedge"])
    total = D_act.sum()
    rows = []
    for name, h in strategies.items():
        cost = realized(h)
        rows.append({"strategy": name, **{f"h_{m}": round(float(h[m]), 4) for m in MONTHS},
                     "realized_cost": cost, "$per_MMBtu": cost / total,
                     "savings_vs_no_hedge": base - cost})
    months = pd.DataFrame({
        "futures_F_m": inst.F, "actual_spot": P_act, "spot_minus_futures": P_act - inst.F,
        "forecast_D_m": inst.D, "actual_demand": D_act, "model_h_m": sol.h,
        "hedged_MMBtu": sol.h * inst.D,
        "hedge_gain_$": sol.h * inst.D * (P_act - inst.F),  # >0 means the hedge paid off
    })
    return Backtest(months, pd.DataFrame(rows).set_index("strategy"), source)

In [ ]:
def report_backtest(cfg: Config, inst: Instance, bt: Backtest) -> None:
    out = out_path(cfg, inst.state)
    print(f"\n--- Backtest: what actually happened in winter {winter_label(cfg.decision_year)} "
          f"({inst.state}) ---")
    print(f"Demand used: {bt.demand_source}")
    print(bt.months.to_string(float_format=lambda v: f"{v:,.3f}" if abs(v) < 100 else f"{v:,.0f}"))
    print()
    print(bt.strategies.drop(columns=[f"h_{m}" for m in MONTHS]).to_string(
        float_format=lambda v: f"{v:,.2f}" if abs(v) < 100 else f"{v:,.0f}"))
    gain = bt.strategies.loc["Model (optimal)", "savings_vs_no_hedge"]
    verdict = "SAVED" if gain > 0 else "COST"
    print(f"\nThe model's hedge {verdict} ${abs(gain):,.0f} versus buying everything at spot.")
    bt.months.to_csv(out / "backtest_months.csv")
    bt.strategies.to_csv(out / "backtest_strategies.csv")
    if cfg.plots:
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            t = bt.strategies["realized_cost"] / 1e6
            fig, ax = plt.subplots(figsize=(8, 4))
            colors = ["#2a6fdb" if n == "Model (optimal)" else "#9aa5b1" for n in t.index]
            ax.barh(t.index, t.values, color=colors)
            for i, v in enumerate(t.values):
                ax.text(v, i, f" {v:,.1f}", va="center", fontsize=9)
            ax.set_xlim(t.min() * 0.95, t.max() * 1.05)
            ax.invert_yaxis()
            ax.set_xlabel("Realized winter cost ($M)")
            ax.set_title(f"{inst.state}: realized cost, winter {winter_label(cfg.decision_year)}")
            fig.tight_layout(); fig.savefig(out / "backtest_costs.png", dpi=150); plt.close(fig)
        except ImportError:
            pass

In [ ]:
def resolve_futures(cfg: Config, fut_hist: pd.DataFrame) -> pd.Series:
    """F_m on the decision date: user input, else the EIA strip for that October if the
    data covers it, else (forecast mode only) the latest October strip with a warning."""
    if cfg.futures is not None:
        return pd.Series(cfg.futures, index=MONTHS, dtype=float)
    lbl = winter_label(cfg.decision_year)
    if lbl in fut_hist.index:
        print(f"  F_m from EIA NYMEX quotes on {fut_hist.attrs['quote_dates'][lbl]}")
        return fut_hist.loc[lbl].copy()
    if actual_winter_spot(cfg, cfg.decision_year) is not None:
        raise ValueError(
            f"Backtesting winter {lbl} needs the NYMEX settlements from Oct 1 {cfg.decision_year} "
            f"(not in the EIA file, which ends Apr 2024). Pass them with --futures NOV DEC JAN FEB "
            f"(or cfg.futures in the notebook).")
    last = fut_hist.index[-1]
    print(f"  NOTE: futures not given; using the latest October strip in the EIA file "
          f"({fut_hist.attrs['quote_dates'][last]}). Pass the Oct 1 {cfg.decision_year} CME "
          f"settlements with --futures NOV DEC JAN FEB.")
    return fut_hist.loc[last].copy()

## 9. Load shared data

In [ ]:
spot = load_spot(cfg)
fut_hist, _ = load_futures(cfg)
F_now = resolve_futures(cfg, fut_hist)   # raises a clear error if a backtest has no futures
print("F_m:", F_now.round(3).to_dict())

hdd = load_hdd(cfg, cfg.states)
gpr = geopolitical_indicator(cfg)
print(f"GPR {gpr_month(cfg)} = {gpr[1]:.1f}, 90th pct threshold = {gpr[2]:.1f} -> g = {gpr[0]}")

In [ ]:
# Historical spot / futures ratio by winter: >1 means spot came in above the October futures
(spot / fut_hist).round(2).style.background_gradient(cmap="RdYlGn_r", axis=None)

## 10. Solve the model for each state (and backtest if the winter is over)

In [ ]:
results, backtests = {}, {}
for st in cfg.states:
    inst = build_instance(cfg, st, spot, fut_hist, F_now, hdd, gpr)
    sol = solve(cfg, inst, resolve_cmax(cfg, inst))
    report(cfg, inst, sol)
    results[st] = (inst, sol)
    bt = backtest(cfg, inst, sol, hdd)
    if bt is not None:
        report_backtest(cfg, inst, bt)
        backtests[st] = bt

## 11. Results tables and charts

In [ ]:
for st, (inst, sol) in results.items():
    print(f"\n=== {st} ===")
    if sol.status != "optimal":
        print("Model", sol.status)
        continue
    display(pd.DataFrame({"hedge share h_m": sol.h.round(4),
                          "hedged volume (MMBtu)": (sol.h * inst.D).round(0),
                          "forecast D_m (MMBtu)": inst.D.round(0),
                          "futures F_m": inst.F}))
    display(strategy_table(inst, sol).style.format(precision=2, thousands=","))
    for png in ["strategy_frontier.png", "sensitivity_heatmap.png"]:
        f = out_path(cfg, st) / png
        if f.exists():
            display(Image(filename=str(f)))

In [ ]:
# Sensitivity matrices (rows: theta, columns: Cmax as a multiple of the 50%-hedge high-price cost)
for st in results:
    d = out_path(cfg, st)
    if (d / "sensitivity_hedge_shares.csv").exists():
        print(f"\n=== {st}: hedge shares Nov/Dec/Jan/Feb ===")
        display(pd.read_csv(d / "sensitivity_hedge_shares.csv", index_col=0))
        print(f"=== {st}: expected cost ($) ===")
        display(pd.read_csv(d / "sensitivity_expected_cost.csv", index_col=0)
                .style.format("{:,.0f}", na_rep="infeasible"))

## 12. Backtest: was the hedge worth it?
Only shown when `decision_year`'s winter is over. `hedge_gain_$` > 0 means locking in futures beat spot that month;
`savings_vs_no_hedge` compares each strategy's realized winter cost with buying everything at spot.

In [ ]:
if not backtests:
    print(f"No backtest: winter {winter_label(cfg.decision_year)} has not happened yet.")
for st, bt in backtests.items():
    print(f"\n=== {st}: winter {winter_label(cfg.decision_year)} (demand: {bt.demand_source}) ===")
    display(bt.months.style.format("{:,.3f}", subset=["futures_F_m", "actual_spot", "spot_minus_futures", "model_h_m"])
            .format("{:,.0f}", subset=["forecast_D_m", "actual_demand", "hedged_MMBtu", "hedge_gain_$"]))
    display(bt.strategies.style.format(precision=2, thousands=","))
    f = out_path(cfg, st) / "backtest_costs.png"
    if f.exists():
        display(Image(filename=str(f)))